# VM14K: EDA và simple ML baseline

Notebook này chạy end-to-end trên **Google Colab (CPU)** với `data/cleaned/clean_final.jsonl`:

1. Đọc và kiểm tra cấu trúc dữ liệu.
2. EDA: nhãn, độ khó, chủ đề, số lựa chọn và độ dài văn bản.
3. Chia train/validation/test theo **nhóm câu hỏi** để câu hỏi trùng không rơi vào hai tập.
4. Train baseline TF-IDF (word + character n-gram) và Logistic Regression.
5. Đánh giá, phân tích lỗi, thử dự đoán và lưu model.

> Đây là baseline nghiên cứu, **không phải hệ thống tư vấn/chẩn đoán y khoa**. Accuracy của mô hình tuyến tính chủ yếu là mốc sanity-check cho các mô hình ngôn ngữ mạnh hơn.

## Cách chạy trên Colab

- Upload notebook này lên Colab và chọn **Runtime → Run all**.
- Không cần upload thủ công dữ liệu: notebook tự `git clone` repo (`REPO_URL` ở cell tìm
  dữ liệu bên dưới) vào `/content/VM14K_Research` nếu chưa có sẵn, rồi đọc thẳng
  `data/cleaned/clean_final.jsonl` và `splits/split_v1.json` từ đó — nhờ vậy phiên bản
  dataset dùng để train luôn khớp với một commit git cụ thể, có thể truy vết.
- Không cần GPU; toàn bộ pipeline thường chỉ mất dưới vài phút trên CPU Colab.
- Nếu đang chạy trong (hoặc cạnh) một checkout local của repo, notebook tự tìm file cục
  bộ và bỏ qua bước clone.

In [ ]:
# 1. Imports và cấu hình
import hashlib
import json
import platform
import random
import subprocess
import sys
import time
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    balanced_accuracy_score,
)
from sklearn.pipeline import FeatureUnion, Pipeline

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
pd.set_option("display.max_colwidth", 120)

print(f"Python: {platform.python_version()}")
print(f"pandas: {pd.__version__} | scikit-learn: {sklearn.__version__}")

In [ ]:
# 2. Tự tìm data trong repo local; nếu không có, git clone repo
# (dataset version == git commit, không cần upload thủ công)
REPO_URL = "https://github.com/hanguyennn2812/VM14K_Research.git"
CLONE_ROOT = Path("/content/VM14K_Research")

candidate_paths = [
    Path("data/cleaned/clean_final.jsonl"),
    Path("../data/cleaned/clean_final.jsonl"),
    CLONE_ROOT / "data" / "cleaned" / "clean_final.jsonl",
]
DATA_PATH = next((p for p in candidate_paths if p.exists()), None)

if DATA_PATH is None:
    if not CLONE_ROOT.exists():
        print(f"Không thấy dữ liệu cục bộ. Đang git clone {REPO_URL} -> {CLONE_ROOT} ...")
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(CLONE_ROOT)],
            check=True,
        )
    DATA_PATH = CLONE_ROOT / "data" / "cleaned" / "clean_final.jsonl"
    if not DATA_PATH.exists():
        raise FileNotFoundError(
            f"Không tìm thấy {DATA_PATH} sau khi clone. Kiểm tra REPO_URL / kết nối mạng."
        )
    REPO_ROOT = CLONE_ROOT
else:
    # Đang chạy trong (hoặc cạnh) một checkout local của repo.
    REPO_ROOT = DATA_PATH.resolve().parents[2]

print("Đang dùng data:", DATA_PATH.resolve())
print("Repo root:", REPO_ROOT.resolve())

In [ ]:
# 3. Đọc JSONL và kiểm tra schema/invariant quan trọng
records, bad_lines = [], []
with DATA_PATH.open("r", encoding="utf-8") as f:
    for line_no, line in enumerate(f, start=1):
        if not line.strip():
            continue
        try:
            records.append(json.loads(line))
        except json.JSONDecodeError as exc:
            bad_lines.append((line_no, str(exc)))

if bad_lines:
    raise ValueError(f"Có dòng JSON lỗi, ví dụ: {bad_lines[:3]}")

df = pd.DataFrame(records)
required_columns = {
    "id", "difficulty_level", "medical_topic", "question",
    "options", "answer", "answer_index", "contradiction_pending_review"
}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f"Thiếu cột bắt buộc: {sorted(missing_columns)}")

df["answer_index"] = df["answer_index"].astype(int)
df["n_options"] = df["options"].map(len)
df["answer_letter"] = df["answer_index"].map(lambda x: chr(65 + x))
df["contradiction_pending_review"] = df["contradiction_pending_review"].astype(bool)

assert df["id"].notna().all(), "Có id rỗng"
assert df["id"].is_unique, "id không unique"
assert df["question"].fillna("").str.strip().ne("").all(), "Có câu hỏi rỗng"
assert df["n_options"].ge(2).all(), "Có câu có ít hơn 2 phương án"
assert (df["answer_index"] < df["n_options"]).all(), "answer_index nằm ngoài options"

print(f"Đọc thành công {len(df):,} câu hỏi; không có dòng JSON lỗi.")
print(f"contradiction_pending_review=true: {int(df['contradiction_pending_review'].sum()):,} dòng "
      "(chờ bác sĩ xác nhận đáp án — xem BUG-2, luôn ở train, không vào test).")
display(df.head(3))

## EDA

Trước khi train, ta kiểm tra missing value, trùng lặp, phân bố nhãn và độ phủ chủ đề. Đặc biệt, VM14K không chỉ có câu 4 lựa chọn; vì vậy metric ngẫu nhiên phải tính theo `1 / số lựa chọn` của từng câu.

In [ ]:
# 4. Tổng quan chất lượng dữ liệu
full_text_key = df.apply(
    lambda r: " || ".join([str(r["question"]), *map(str, r["options"])]), axis=1
)
overview = pd.Series({
    "Số câu": len(df),
    "Số cột": df.shape[1],
    "ID unique": df["id"].nunique(),
    "Câu hỏi unique": df["question"].nunique(),
    "Toàn bộ question+options unique": full_text_key.nunique(),
    "Tổng ô missing": int(df.isna().sum().sum()),
    "Số chủ đề unique": df["medical_topic"].explode().nunique(),
})
display(overview.to_frame("Giá trị"))

missing = df.isna().sum().sort_values(ascending=False)
display(missing[missing.gt(0)].to_frame("missing") if missing.gt(0).any()
        else pd.DataFrame({"Kết quả": ["Không có missing value"]}))

In [ ]:
# 5. Các phân bố chính
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

df["answer_letter"].value_counts().sort_index().plot.bar(
    ax=axes[0, 0], color="#315b7d", title="Phân bố đáp án đúng"
)
axes[0, 0].set(xlabel="Vị trí đáp án", ylabel="Số câu")

# Thứ tự độ khó của paper: Easy < Medium < Challenging < Hard (BUG-5).
difficulty_order = ["Easy", "Medium", "Challenging", "Hard"]
df["difficulty_level"].value_counts().reindex(difficulty_order).dropna().plot.bar(
    ax=axes[0, 1], color="#bd6b4d", title="Phân bố độ khó"
)
axes[0, 1].set(xlabel="Độ khó", ylabel="Số câu")

df["n_options"].value_counts().sort_index().plot.bar(
    ax=axes[1, 0], color="#4e8b78", title="Số phương án mỗi câu"
)
axes[1, 0].set(xlabel="Số phương án", ylabel="Số câu")

top_topics = df["medical_topic"].explode().value_counts().head(12).sort_values()
top_topics.plot.barh(
    ax=axes[1, 1], color="#8b6fa8", title="12 chủ đề xuất hiện nhiều nhất"
)
axes[1, 1].set(xlabel="Số lần xuất hiện", ylabel="")

plt.tight_layout()
plt.show()

In [ ]:
# 6. Độ dài văn bản
df["question_words"] = df["question"].str.split().map(len)
df["mean_option_words"] = df["options"].map(
    lambda opts: float(np.mean([len(str(opt).split()) for opt in opts]))
)

display(
    df[["question_words", "mean_option_words", "n_options"]]
    .describe(percentiles=[0.5, 0.9, 0.95, 0.99])
    .round(2)
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
df["question_words"].clip(upper=df["question_words"].quantile(0.99)).plot.hist(
    bins=40, ax=axes[0], color="#315b7d", title="Số từ trong câu hỏi (clip p99)"
)
df["mean_option_words"].clip(upper=df["mean_option_words"].quantile(0.99)).plot.hist(
    bins=40, ax=axes[1], color="#bd6b4d", title="Số từ trung bình/phương án (clip p99)"
)
plt.tight_layout()
plt.show()

In [ ]:
# 7. Xem mẫu ngẫu nhiên ở dạng dễ đọc
def readable_sample(row):
    lines = [f"Q: {row['question']}"]
    lines.extend(f"  {chr(65+i)}. {opt}" for i, opt in enumerate(row["options"]))
    lines.append(f"Đáp án: {row['answer_letter']} | Độ khó: {row['difficulty_level']}")
    return "\n".join(lines)

for _, sample_row in df.sample(3, random_state=SEED).iterrows():
    print(readable_sample(sample_row))
    print("-" * 100)

## Thiết kế bài toán và chia tập

Ta dùng **candidate ranking** để hỗ trợ tự nhiên các câu có 2–5 phương án:

- Mỗi phương án trở thành một sample: `(câu hỏi, phương án, vị trí)`.
- Nhãn bằng 1 nếu phương án đúng, ngược lại bằng 0.
- Khi dự đoán, model chấm mọi phương án của một câu và chọn điểm cao nhất.

Một linear bag-of-words model không thực sự hiểu kiến thức y khoa hay tương tác ngữ nghĩa sâu giữa câu hỏi và phương án. Đây là baseline rẻ, nhanh và tái lập được.

**Split train/val/test đã được đóng băng từ trước** trong `splits/split_v1.json` (sinh bởi
`scripts/analysis/gen_split.py`) — notebook này chỉ **load theo `id`**, không tính lại.
Split đó nhóm theo `question` đã chuẩn hoá bằng normaliser của các script phân tích
(`dedup_utils.normalize_vietnamese`, không chỉ `casefold()` — xem BUG-4) để cùng một stem
không rơi vào 2 split khác nhau, và đưa toàn bộ các dòng `contradiction_pending_review=true`
(đáp án chưa được bác sĩ xác nhận, xem BUG-2) về train, không bao giờ vào test.

In [ ]:
# 8. Load split train/val/test đã đóng băng (splits/split_v1.json) — KHÔNG tính lại ở đây
sys.path.insert(0, str(REPO_ROOT / "scripts" / "analysis"))
from dedup_utils import normalize_vietnamese

SPLIT_PATH = REPO_ROOT / "splits" / "split_v1.json"
with SPLIT_PATH.open("r", encoding="utf-8") as f:
    split_by_id = json.load(f)

model_df = df.copy()
missing_split = set(model_df["id"]) - set(split_by_id)
if missing_split:
    raise ValueError(
        f"{len(missing_split)} id không có trong {SPLIT_PATH.name}, "
        f"vd: {sorted(missing_split)[:5]}. Chạy lại scripts/analysis/gen_split.py."
    )

model_df["split"] = model_df["id"].map(split_by_id)
train_df = model_df.loc[model_df["split"] == "train"].reset_index(drop=True)
val_df = model_df.loc[model_df["split"] == "val"].reset_index(drop=True)
test_df = model_df.loc[model_df["split"] == "test"].reset_index(drop=True)

split_summary = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "questions": [len(train_df), len(val_df), len(test_df)],
})
split_summary["percent"] = (100 * split_summary["questions"] / len(model_df)).round(2)
display(split_summary)

# Re-verify (không tin mù quáng) các bất biến mà split_v1.json phải giữ, dùng đúng
# normaliser đã dùng để tạo split (BUG-4: casefold() không đủ, phải chuẩn hoá dấu +
# dấu câu tiếng Việt).
model_df["question_group"] = model_df["question"].map(normalize_vietnamese)
stems_per_split = model_df.groupby("question_group")["split"].nunique()
leaking_stems = stems_per_split[stems_per_split > 1]
assert leaking_stems.empty, (
    f"{len(leaking_stems)} normalised question stem xuất hiện ở nhiều hơn 1 split "
    f"(vd: {leaking_stems.index[:3].tolist()})"
)
print("OK: không có question stem nào (đã chuẩn hoá) xuất hiện ở nhiều hơn 1 split.")

leaked_pending = int(test_df["contradiction_pending_review"].sum())
assert leaked_pending == 0, f"{leaked_pending} contradiction_pending_review=true row lọt vào test"
print("OK: không có contradiction_pending_review=true nào trong test.")

In [ ]:
# 9. Chuyển mỗi câu thành các candidate samples
# EDA cho thấy phân bố đáp án không đều (đáp án A ~30%, xem baseline always-first-option
# ở dưới). Mặc định KHÔNG đưa vị trí A/B/C/... vào feature: baseline always-first-option
# đã đo sẵn tín hiệu vị trí đó rồi, cho model học lại nó là double-count một lỗi của
# dataset, không phải kiến thức y khoa (BUG-3). Đặt True nếu muốn thử biến thể
# content+position — khi đó hãy chạy CẢ HAI và luôn báo cáo content-only là con số chính.
USE_POSITION_FEATURE = False
POSITION_LABEL = "content+position" if USE_POSITION_FEATURE else "content-only"

def make_candidates(frame, include_label=True):
    rows = []
    for row in frame.itertuples(index=False):
        for option_index, option_text in enumerate(row.options):
            position_token = (
                f"VI_TRI_{chr(65 + option_index)} " if USE_POSITION_FEATURE else ""
            )
            candidate_text = (
                f"{position_token}Câu hỏi: {row.question} "
                f"[SEP] Phương án: {option_text}"
            )
            item = {
                "id": row.id,
                "option_index": option_index,
                "candidate_text": candidate_text,
            }
            if include_label:
                item["label"] = int(option_index == row.answer_index)
            rows.append(item)
    return pd.DataFrame(rows)

train_candidates = make_candidates(train_df)
print(f"{len(train_df):,} câu train → {len(train_candidates):,} candidate samples")
display(train_candidates.head())
display(train_candidates["label"].value_counts().rename_axis("label").to_frame("count"))

## Baseline không học

Ta so model với hai mốc:

- **Random hợp lệ:** chọn ngẫu nhiên trong đúng số phương án của từng câu; kỳ vọng là trung bình `1/n_options`.
- **Always A:** luôn chọn phương án đầu tiên. Đây là mốc quan trọng vì đáp án A xuất hiện nhiều hơn các vị trí khác.

In [ ]:
# 10. Baseline metrics trên validation (chưa nhìn test)
def no_skill_baselines(frame):
    return {
        "Random hợp lệ (expected)": float(np.mean(1.0 / frame["n_options"])),
        "Always A": float(np.mean(frame["answer_index"].eq(0))),
    }

display(
    pd.Series(no_skill_baselines(val_df), name="validation_accuracy")
    .to_frame()
    .style.format("{:.3%}")
)

## Train TF-IDF + Logistic Regression

- Word n-gram `(1, 2)` nắm từ/cụm từ ngắn.
- Character n-gram `(3, 5)` bền hơn với biến thể chính tả và từ chuyên ngành.
- Logistic Regression chấm xác suất đúng của từng candidate.

Giới hạn `max_features` giữ RAM và thời gian phù hợp Colab CPU.

In [ ]:
# 11. Khởi tạo và train model
model = Pipeline([
    ("features", FeatureUnion([
        ("word", TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=2,
            max_features=40_000,
            sublinear_tf=True,
        )),
        ("char", TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            min_df=3,
            max_features=40_000,
            sublinear_tf=True,
        )),
    ])),
    ("classifier", LogisticRegression(
        C=2.0,
        solver="liblinear",
        max_iter=500,
        random_state=SEED,
    )),
])

start = time.perf_counter()
model.fit(train_candidates["candidate_text"], train_candidates["label"])
elapsed = time.perf_counter() - start
print(f"Train xong trong {elapsed:.1f} giây.")

In [ ]:
# 12. Hàm chấm từng candidate và chọn phương án tốt nhất của mỗi câu
def predict_questions(fitted_model, frame):
    candidates = make_candidates(frame, include_label=False)
    candidates["raw_score"] = fitted_model.predict_proba(
        candidates["candidate_text"]
    )[:, 1]
    score_sum = candidates.groupby("id")["raw_score"].transform("sum").clip(lower=1e-12)
    candidates["relative_score"] = candidates["raw_score"] / score_sum

    best_rows = candidates.loc[
        candidates.groupby("id")["raw_score"].idxmax(),
        ["id", "option_index", "relative_score"],
    ].rename(columns={
        "option_index": "pred_index",
        "relative_score": "confidence",
    })

    keep = [
        "id", "question", "options", "answer_index", "answer_letter",
        "difficulty_level", "medical_topic", "n_options"
    ]
    result = frame[keep].merge(best_rows, on="id", how="left", validate="one_to_one")
    result["pred_index"] = result["pred_index"].astype(int)
    result["pred_letter"] = result["pred_index"].map(lambda x: chr(65 + x))
    result["correct"] = result["pred_index"].eq(result["answer_index"])
    return result, candidates

def metric_summary(result):
    return {
        "accuracy": accuracy_score(result["answer_index"], result["pred_index"]),
        "balanced_accuracy_by_position": balanced_accuracy_score(
            result["answer_index"], result["pred_index"]
        ),
    }

val_result, val_candidate_scores = predict_questions(model, val_df)
display(pd.Series(metric_summary(val_result), name="validation").to_frame().style.format("{:.3%}"))

## Đánh giá cuối trên test

Chỉ sau khi pipeline đã cố định bằng validation, ta mới đo test. Với baseline đơn giản này, mục tiêu không phải accuracy cao mà là có một quy trình đúng để các model sau phải vượt qua.

In [ ]:
# 13. Test metrics và so sánh baseline
test_result, test_candidate_scores = predict_questions(model, test_df)
test_metrics = metric_summary(test_result)

comparison = {
    **no_skill_baselines(test_df),
    f"TF-IDF + Logistic Regression ({POSITION_LABEL})": test_metrics["accuracy"],
}
comparison_df = pd.Series(comparison, name="test_accuracy").to_frame()
display(comparison_df.style.format("{:.3%}"))

ax = comparison_df.sort_values("test_accuracy")["test_accuracy"].plot.barh(
    figsize=(8, 3.5), color=["#9aa0a6", "#9aa0a6", "#315b7d"]
)
ax.set(xlabel="Accuracy", ylabel="", title=f"So sánh trên test ({POSITION_LABEL})")
ax.set_xlim(0, max(comparison_df["test_accuracy"].max() * 1.2, 0.4))
for container in ax.containers:
    ax.bar_label(container, fmt="%.3f", padding=3)
plt.tight_layout()
plt.show()

In [ ]:
# 14. Confusion matrix theo vị trí đáp án
all_indices = sorted(set(test_result["answer_index"]) | set(test_result["pred_index"]))
display_labels = [chr(65 + i) for i in all_indices]
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    test_result["answer_index"],
    test_result["pred_index"],
    labels=all_indices,
    display_labels=display_labels,
    normalize="true",
    cmap="Blues",
    values_format=".2f",
    ax=ax,
)
ax.set_title("Confusion matrix chuẩn hóa theo nhãn thật")
plt.tight_layout()
plt.show()

In [ ]:
# 15. Accuracy theo độ khó và theo số phương án
by_difficulty = (
    test_result.groupby("difficulty_level", observed=True)
    .agg(n=("correct", "size"), accuracy=("correct", "mean"))
    .sort_values("n", ascending=False)
)
by_n_options = (
    test_result.groupby("n_options", observed=True)
    .agg(n=("correct", "size"), accuracy=("correct", "mean"))
    .sort_index()
)
print("Theo độ khó:")
display(by_difficulty.style.format({"accuracy": "{:.3%}"}))
print("Theo số phương án:")
display(by_n_options.style.format({"accuracy": "{:.3%}"}))

In [ ]:
# 16. Error analysis: các dự đoán sai mà model tự tin nhất
wrong = test_result.loc[~test_result["correct"]].copy()
wrong["gold_text"] = wrong.apply(lambda r: r["options"][r["answer_index"]], axis=1)
wrong["pred_text"] = wrong.apply(lambda r: r["options"][r["pred_index"]], axis=1)
error_columns = [
    "question", "answer_letter", "gold_text", "pred_letter",
    "pred_text", "confidence", "difficulty_level"
]
display(
    wrong.sort_values("confidence", ascending=False)[error_columns]
    .head(12)
    .style.format({"confidence": "{:.2%}"})
)

In [ ]:
# 17. Feature associations (không nên diễn giải như kiến thức y khoa)
feature_names = model.named_steps["features"].get_feature_names_out()
coefficients = model.named_steps["classifier"].coef_[0]
top_k = 20
top_positive_idx = np.argsort(coefficients)[-top_k:][::-1]
top_negative_idx = np.argsort(coefficients)[:top_k]

feature_table = pd.DataFrame({
    "Liên hệ với candidate đúng": feature_names[top_positive_idx],
    "Hệ số (+)": coefficients[top_positive_idx],
    "Liên hệ với candidate sai": feature_names[top_negative_idx],
    "Hệ số (-)": coefficients[top_negative_idx],
})
display(feature_table.round(4))

## Thử dự đoán một câu mới

Hàm dưới đây dùng đúng quy trình candidate ranking. `score` chỉ nên dùng để xếp hạng các lựa chọn trong cùng một câu, không phải độ tin cậy y khoa đã được calibration.

In [ ]:
# 18. Inference helper
def predict_one(question, options, fitted_model=model):
    if len(options) < 2:
        raise ValueError("Cần ít nhất 2 phương án.")
    texts = []
    for i, option in enumerate(options):
        position_token = f"VI_TRI_{chr(65 + i)} " if USE_POSITION_FEATURE else ""
        texts.append(
            f"{position_token}Câu hỏi: {question} [SEP] Phương án: {option}"
        )
    raw_scores = fitted_model.predict_proba(texts)[:, 1]
    relative_scores = raw_scores / np.clip(raw_scores.sum(), 1e-12, None)
    result = pd.DataFrame({
        "letter": [chr(65 + i) for i in range(len(options))],
        "option": options,
        "score": relative_scores,
    }).sort_values("score", ascending=False)
    return result

# Lấy một câu test để kiểm tra luồng inference
example = test_df.iloc[0]
print(example["question"])
display(predict_one(example["question"], example["options"]).style.format({"score": "{:.2%}"}))
print("Nhãn thật:", example["answer_letter"])

## Lưu artifact

Artifact chứa TF-IDF vocabulary, Logistic Regression và cờ feature vị trí. Chỉ load file `joblib` do chính mình tạo hoặc từ nguồn tin cậy.

Cả artifact và `vm14k_test_metrics.json` đều được gắn **provenance**: sha256 của
`clean_final.jsonl`, tên file split, git commit của repo, và `USE_POSITION_FEATURE` — để
biết chính xác dataset/commit nào tạo ra con số này.

In [ ]:
# 19. Provenance (dataset sha256, split, git commit, cờ vị trí) + lưu model/metrics
def git_commit_hash(repo_root: Path) -> str:
    try:
        out = subprocess.run(
            ["git", "-C", str(repo_root), "rev-parse", "HEAD"],
            capture_output=True, text=True, check=True,
        )
        return out.stdout.strip()
    except Exception as exc:
        return f"unknown ({exc})"

dataset_sha256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()

clean_report_path = REPO_ROOT / "reports" / "cleaning" / "clean_final.report.json"
if clean_report_path.exists():
    reported_sha256 = json.loads(
        clean_report_path.read_text(encoding="utf-8")
    )["output"]["sha256"]
    if reported_sha256 != dataset_sha256:
        print(
            f"CẢNH BÁO: sha256 tính trực tiếp từ {DATA_PATH.name} khác với "
            f"clean_final.report.json ({reported_sha256} != {dataset_sha256}) — "
            "dataset có thể đã bị sửa sau khi report được sinh ra."
        )
else:
    print(f"Không thấy {clean_report_path}; chỉ dùng sha256 tính trực tiếp.")

provenance = {
    "dataset_path": str(DATA_PATH),
    "dataset_sha256": dataset_sha256,
    "split_file": SPLIT_PATH.name,
    "git_commit": git_commit_hash(REPO_ROOT),
    "use_position_feature": USE_POSITION_FEATURE,
}

ARTIFACT_PATH = Path("vm14k_tfidf_logreg.joblib")
METRICS_PATH = Path("vm14k_test_metrics.json")

artifact = {
    "model": model,
    "use_position_feature": USE_POSITION_FEATURE,
    "seed": SEED,
    "task": "candidate ranking for variable-choice Vietnamese medical MCQ",
    "provenance": provenance,
}
joblib.dump(artifact, ARTIFACT_PATH)

metrics_to_save = {
    "n_train_questions": len(train_df),
    "n_validation_questions": len(val_df),
    "n_test_questions": len(test_df),
    "random_expected_accuracy": no_skill_baselines(test_df)["Random hợp lệ (expected)"],
    "always_a_accuracy": no_skill_baselines(test_df)["Always A"],
    **test_metrics,
    "provenance": provenance,
}
METRICS_PATH.write_text(
    json.dumps(metrics_to_save, ensure_ascii=False, indent=2), encoding="utf-8"
)

print(f"Đã lưu: {ARTIFACT_PATH} ({ARTIFACT_PATH.stat().st_size / 1e6:.1f} MB)")
print(f"Đã lưu: {METRICS_PATH}")
print(json.dumps(metrics_to_save, ensure_ascii=False, indent=2))

# Trên Colab, bỏ comment 2 dòng dưới nếu muốn tải file về máy:
# from google.colab import files
# files.download(str(ARTIFACT_PATH))

## Kết luận và bước tiếp theo

- TF-IDF + Logistic Regression là baseline nhanh, dễ debug và phù hợp để kiểm tra pipeline.
- Do model tuyến tính không hiểu sâu quan hệ giữa stem và đáp án, accuracy sẽ còn thấp. Không dùng model này trong quyết định y khoa.
- Nên báo cáo cả random baseline, Always-A baseline, accuracy theo độ khó/số phương án và error analysis — không chỉ một con số tổng.
- Bước tiếp theo hợp lý: fine-tune một multilingual Transformer theo multiple-choice/candidate-ranking, giữ nguyên group split và test set trong notebook này để so sánh công bằng.
- Trước một benchmark nghiêm túc hơn, nên tạo thêm split theo nhóm near-duplicate hoặc theo nguồn/chủ đề nếu metadata nguồn sẵn có.